In [0]:

from datetime import datetime, timezone
from pyspark.sql import functions as F

RUN_ID = "run_20261026T150532Z_ad1ccc"

BASE_PATH = "/Volumes/workspace/default/smart_energy_landing"

BRONZE_RUN_PATH = f"{BASE_PATH}/bronze/historical/run_id={RUN_ID}"
BRONZE_READINGS_PATH = f"{BRONZE_RUN_PATH}/readings"
REFERENCE_PATH = f"{BRONZE_RUN_PATH}/reference"

SILVER_RUN_PATH = f"{BASE_PATH}/silver/historical/run_id={RUN_ID}"
QUARANTINE_RUN_PATH = f"{BASE_PATH}/quarantine/historical/run_id={RUN_ID}"

# Historical data contract for this run.
EXPECTED_START_DATE = "2026-07-01"
EXPECTED_END_DATE_EXCLUSIVE = "2026-07-07"

EXPECTED_SCHEMA_VERSION = 1
READING_INTERVAL_MINUTES = 15
NOMINAL_VOLTAGE_V = 220.0
LOW_VOLTAGE_LIMIT_V = 207.0
HIGH_VOLTAGE_LIMIT_V = 253.0
ENERGY_TOLERANCE_KWH = 0.02

PIPELINE_STARTED_AT_UTC = datetime.now(timezone.utc)

spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Run ID:", RUN_ID)
print("Bronze:", BRONZE_RUN_PATH)
print("Silver:", SILVER_RUN_PATH)
print("Quarantine:", QUARANTINE_RUN_PATH)


In [0]:
#Read Bronze and Reference Data

bronze_df = spark.read.parquet(BRONZE_READINGS_PATH)

dim_meter_df = spark.read.parquet(
    f"{REFERENCE_PATH}/dim_meter"
)

dim_transformer_df = spark.read.parquet(
    f"{REFERENCE_PATH}/dim_transformer"
)

dim_substation_df = spark.read.parquet(
    f"{REFERENCE_PATH}/dim_substation"
)

dim_region_df = spark.read.parquet(
    f"{REFERENCE_PATH}/dim_region"
)

print("Bronze readings:", f"{bronze_df.count():,}")
print("Meters:", f"{dim_meter_df.count():,}")
print("Transformers:", f"{dim_transformer_df.count():,}")
print("Substations:", f"{dim_substation_df.count():,}")
print("Regions:", f"{dim_region_df.count():,}")

display(bronze_df.limit(5))


In [0]:
#Prepare Readings and Normalize Types

readings_prepared_df = (
    bronze_df
    .withColumnRenamed("reading_id", "reading_id_raw")
    .withColumnRenamed("event_timestamp", "event_timestamp_raw")
    .withColumnRenamed("meter_id", "meter_id_raw")
    .withColumnRenamed("transformer_id", "transformer_id_raw")
    .withColumn("reading_id", F.trim("reading_id_raw"))
    .withColumn("meter_id", F.trim("meter_id_raw"))
    .withColumn("transformer_id", F.trim("transformer_id_raw"))
    .withColumn(
        "event_timestamp",
        F.expr("try_to_timestamp(trim(event_timestamp_raw))")
    )
    .withColumn(
        "pipeline_processed_at_utc",
        F.lit(PIPELINE_STARTED_AT_UTC.replace(tzinfo=None))
         .cast("timestamp")
    )
    .withColumn("year", F.year("event_timestamp"))
    .withColumn("month", F.month("event_timestamp"))
    .withColumn("day", F.dayofmonth("event_timestamp"))
)

display(readings_prepared_df.limit(5))


In [0]:
#Validate Required Fields, Numeric Values and Timestamps

start_timestamp = F.to_timestamp(F.lit(EXPECTED_START_DATE))
end_timestamp = F.to_timestamp(F.lit(EXPECTED_END_DATE_EXCLUSIVE))

# Detect NaN and positive/negative infinity in numeric columns.
numeric_columns = [
    "voltage_v",
    "active_power_kw",
    "reactive_power_kvar",
    "energy_kwh",
    "power_factor"
]

numeric_invalid_conditions = []

for column_name in numeric_columns:
    numeric_invalid_conditions.append(
        F.when(
            F.col(column_name).isNotNull()
            & (
                F.isnan(F.col(column_name))
                | (F.abs(F.col(column_name)) == float("inf"))
            ),
            F.lit(f"NON_FINITE_{column_name.upper()}")
        )
    )

hard_error_conditions = [
    F.when(
        F.col("reading_id").isNull()
        | (F.col("reading_id") == ""),
        F.lit("MISSING_READING_ID")
    ),
    F.when(
        F.col("event_timestamp").isNull(),
        F.lit("INVALID_EVENT_TIMESTAMP")
    ),
    F.when(
        F.col("meter_id").isNull() | (F.col("meter_id") == ""),
        F.lit("MISSING_METER_ID")
    ),
    F.when(
        F.col("transformer_id").isNull()
        | (F.col("transformer_id") == ""),
        F.lit("MISSING_TRANSFORMER_ID")
    ),
    F.when(
        F.col("voltage_v").isNull(),
        F.lit("MISSING_VOLTAGE")
    ),
    F.when(
        F.col("active_power_kw").isNull(),
        F.lit("MISSING_ACTIVE_POWER")
    ),
    F.when(
        F.col("energy_kwh").isNull(),
        F.lit("MISSING_ENERGY")
    ),
    F.when(
        F.col("voltage_v") <= 0,
        F.lit("INVALID_VOLTAGE")
    ),
    F.when(
        F.col("active_power_kw") < 0,
        F.lit("NEGATIVE_ACTIVE_POWER")
    ),
    F.when(
        F.col("energy_kwh") < 0,
        F.lit("NEGATIVE_ENERGY")
    ),
    F.when(
        F.col("schema_version").isNull()
        | (F.col("schema_version") != EXPECTED_SCHEMA_VERSION),
        F.lit("INVALID_SCHEMA_VERSION")
    ),
    F.when(
        F.col("event_timestamp").isNotNull()
        & (
            (F.col("event_timestamp") < start_timestamp)
            | (F.col("event_timestamp") >= end_timestamp)
        ),
        F.lit("TIMESTAMP_OUTSIDE_RUN_RANGE")
    ),
    F.when(
        F.col("event_timestamp").isNotNull()
        & (
            (F.minute("event_timestamp") % READING_INTERVAL_MINUTES != 0)
            | (F.second("event_timestamp") != 0)
        ),
        F.lit("TIMESTAMP_NOT_ALIGNED_TO_15_MIN")
    )
]

hard_error_conditions.extend(numeric_invalid_conditions)

quality_flag_conditions = [
    F.when(
        F.col("voltage_v").between(0, LOW_VOLTAGE_LIMIT_V - 0.000001),
        F.lit("LOW_VOLTAGE")
    ),
    F.when(
        F.col("voltage_v") > HIGH_VOLTAGE_LIMIT_V,
        F.lit("HIGH_VOLTAGE")
    ),
    F.when(
        F.col("power_factor").isNotNull()
        & (
            (F.col("power_factor") < 0)
            | (F.col("power_factor") > 1)
        ),
        F.lit("POWER_FACTOR_OUT_OF_RANGE")
    ),
    F.when(
        F.col("energy_kwh").isNotNull()
        & F.col("active_power_kw").isNotNull()
        & (F.col("energy_kwh") >= 0)
        & (F.col("active_power_kw") >= 0)
        & (
            F.abs(
                F.col("energy_kwh")
                - F.col("active_power_kw") * F.lit(0.25)
            ) > ENERGY_TOLERANCE_KWH
        ),
        F.lit("ENERGY_CALCULATION_MISMATCH")
    )
]

readings_quality_df = (
    readings_prepared_df
    .withColumn(
        "validation_errors",
        F.filter(
            F.array(*hard_error_conditions),
            lambda x: x.isNotNull()
        )
    )
    .withColumn(
        "quality_flags",
        F.filter(
            F.array(*quality_flag_conditions),
            lambda x: x.isNotNull()
        )
    )
    .withColumn(
        "record_status",
        F.when(
            F.size("validation_errors") > 0,
            F.lit("QUARANTINE")
        ).otherwise(F.lit("ACCEPT"))
    )
)

display(
    readings_quality_df.select(
        "reading_id",
        "event_timestamp",
        "voltage_v",
        "active_power_kw",
        "energy_kwh",
        "validation_errors",
        "quality_flags",
        "record_status"
    ).limit(10)
)


In [0]:
#Enrich Readings with Reference Dimensions

meter_ref = dim_meter_df.select(
    F.trim("meter_id").alias("ref_meter_id"),
    F.trim("transformer_id").alias("meter_transformer_id"),
    "customer_type",
    "tariff_category",
    "phase_type",
    F.col("status").alias("meter_status")
)

transformer_ref = dim_transformer_df.select(
    F.trim("transformer_id").alias("ref_transformer_id"),
    F.trim("substation_id").alias("substation_id"),
    F.trim("region_id").alias("transformer_region_id"),
    "rated_capacity_kva",
    "nominal_voltage_v",
    F.col("status").alias("transformer_status")
)

substation_ref = dim_substation_df.select(
    F.trim("substation_id").alias("ref_substation_id"),
    F.col("name").alias("substation_name"),
    F.trim("region_id").alias("substation_region_id"),
    F.col("city").alias("substation_city"),
    "primary_kv",
    "secondary_kv",
    "capacity_mva",
    F.col("status").alias("substation_status")
)

region_ref = dim_region_df.select(
    F.trim("region_id").alias("ref_region_id"),
    F.col("region_name")
)

readings_enriched_df = (
    readings_quality_df.alias("r")
    .join(
        meter_ref.alias("m"),
        F.col("r.meter_id") == F.col("m.ref_meter_id"),
        "left"
    )
    .join(
        transformer_ref.alias("t"),
        F.col("r.transformer_id") == F.col("t.ref_transformer_id"),
        "left"
    )
    .join(
        substation_ref.alias("s"),
        F.col("t.substation_id") == F.col("s.ref_substation_id"),
        "left"
    )
    .join(
        region_ref.alias("g"),
        F.col("t.transformer_region_id") == F.col("g.ref_region_id"),
        "left"
    )
    .withColumn(
        "reference_errors",
        F.filter(
            F.array(
                F.when(
                    F.col("m.ref_meter_id").isNull(),
                    F.lit("UNKNOWN_METER_ID")
                ),
                F.when(
                    F.col("t.ref_transformer_id").isNull(),
                    F.lit("UNKNOWN_TRANSFORMER_ID")
                ),
                F.when(
                    F.col("s.ref_substation_id").isNull(),
                    F.lit("UNKNOWN_SUBSTATION_ID")
                ),
                F.when(
                    F.col("g.ref_region_id").isNull(),
                    F.lit("UNKNOWN_REGION_ID")
                ),
                F.when(
                    F.col("m.ref_meter_id").isNotNull()
                    & F.col("t.ref_transformer_id").isNotNull()
                    & (
                        F.col("m.meter_transformer_id")
                        != F.col("r.transformer_id")
                    ),
                    F.lit("METER_TRANSFORMER_MISMATCH")
                ),
                F.when(
                    F.col("s.substation_region_id").isNotNull()
                    & F.col("t.transformer_region_id").isNotNull()
                    & (
                        F.col("s.substation_region_id")
                        != F.col("t.transformer_region_id")
                    ),
                    F.lit("SUBSTATION_REGION_MISMATCH")
                )
            ),
            lambda x: x.isNotNull()
        )
    )
    .withColumn(
        "validation_errors",
        F.concat(
            F.col("r.validation_errors"),
            F.col("reference_errors")
        )
    )
    .withColumn(
        "record_status",
        F.when(
            F.size("validation_errors") > 0,
            F.lit("QUARANTINE")
        ).otherwise(F.lit("ACCEPT"))
    )
    .drop(
        "ref_meter_id",
        "meter_transformer_id",
        "ref_transformer_id",
        "ref_substation_id",
        "ref_region_id",
        "reference_errors"
    )
)

display(readings_enriched_df.limit(5))


In [0]:
#Deduplicate and Detect Conflicting Reading IDs

reading_identity_columns = [
    "reading_id",
    "event_timestamp",
    "meter_id",
    "transformer_id",
    "voltage_v",
    "active_power_kw",
    "reactive_power_kvar",
    "energy_kwh",
    "power_factor",
    "schema_version"
]

with_payload_hash_df = readings_enriched_df.withColumn(
    "payload_hash",
    F.sha2(
        F.to_json(
            F.struct(
                *[F.col(c) for c in reading_identity_columns]
            )
        ),
        256
    )
)

conflicting_ids_df = (
    with_payload_hash_df
    .groupBy("reading_id")
    .agg(
        F.countDistinct("payload_hash").alias("distinct_payloads")
    )
    .filter(F.col("distinct_payloads") > 1)
    .select("reading_id")
    .withColumn("has_conflicting_duplicate", F.lit(True))
)

deduplicated_enriched_df = (
    with_payload_hash_df
    .join(conflicting_ids_df, on="reading_id", how="left")
    .withColumn(
        "validation_errors",
        F.when(
            F.col("has_conflicting_duplicate") == True,
            F.concat(
                F.col("validation_errors"),
                F.array(F.lit("CONFLICTING_DUPLICATE_READING_ID"))
            )
        ).otherwise(F.col("validation_errors"))
    )
    .withColumn(
        "record_status",
        F.when(
            F.size("validation_errors") > 0,
            F.lit("QUARANTINE")
        ).otherwise(F.lit("ACCEPT"))
    )
    .drop("payload_hash", "has_conflicting_duplicate")
    .dropDuplicates(reading_identity_columns)
)

print("Rows before deduplication:", f"{readings_enriched_df.count():,}")
print("Rows after deduplication:", f"{deduplicated_enriched_df.count():,}")

display(
    deduplicated_enriched_df
    .groupBy("record_status")
    .count()
)


In [0]:
#Apply Row-Level Silver Transformations

silver_transformed_df = (
    deduplicated_enriched_df
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("event_hour", F.hour("event_timestamp"))
    .withColumn(
        "day_of_week",
        F.date_format("event_timestamp", "EEEE")
    )
    .withColumn(
        "energy_kwh_calculated",
        F.round(
            F.col("active_power_kw") * F.lit(0.25),
            4
        )
    )
    .withColumn(
        "voltage_deviation_v",
        F.round(
            F.col("voltage_v") - F.lit(NOMINAL_VOLTAGE_V),
            2
        )
    )
    .withColumn("event_year", F.year("event_timestamp"))
    .withColumn("event_month", F.month("event_timestamp"))
    .withColumn("event_day", F.dayofmonth("event_timestamp"))
)

display(silver_transformed_df.limit(10))


In [0]:
#Separate Silver and Quarantine

raw_helper_columns = [
    "reading_id_raw",
    "event_timestamp_raw",
    "meter_id_raw",
    "transformer_id_raw"
]

silver_final_df = (
    silver_transformed_df
    .filter(F.col("record_status") == "ACCEPT")
    .drop(*raw_helper_columns)
)

quarantine_final_df = (
    silver_transformed_df
    .filter(F.col("record_status") == "QUARANTINE")
    .drop(*raw_helper_columns)
)

print("Silver rows:", f"{silver_final_df.count():,}")
print("Quarantine rows:", f"{quarantine_final_df.count():,}")

display(
    silver_final_df.select(
        "reading_id",
        "event_timestamp",
        "meter_id",
        "transformer_id",
        "voltage_v",
        "energy_kwh",
        "energy_kwh_calculated",
        "voltage_deviation_v",
        "quality_flags",
        "record_status"
    ).limit(10)
)


In [0]:
#Review Silver and Quarantine Before Saving

print("Silver schema:")
silver_final_df.printSchema()

print("Silver rows:", f"{silver_final_df.count():,}")
print("Quarantine rows:", f"{quarantine_final_df.count():,}")

display(
    silver_final_df.agg(
        F.count("*").alias("total_readings"),
        F.countDistinct("reading_id").alias("distinct_reading_ids"),
        F.countDistinct("meter_id").alias("distinct_meters"),
        F.min("event_timestamp").alias("first_event_timestamp"),
        F.max("event_timestamp").alias("last_event_timestamp"),
        F.sum(
            F.when(
                F.array_contains("quality_flags", "LOW_VOLTAGE"),
                1
            ).otherwise(0)
        ).alias("low_voltage_readings"),
        F.sum(
            F.when(
                F.array_contains(
                    "quality_flags",
                    "ENERGY_CALCULATION_MISMATCH"
                ),
                1
            ).otherwise(0)
        ).alias("energy_mismatch_readings")
    )
)

display(
    quarantine_final_df
    .groupBy("validation_errors")
    .count()
    .orderBy(F.desc("count"))
)


In [0]:
#Save Silver and Quarantine as Delta

(
    silver_final_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("event_year", "event_month", "event_day")
    .save(SILVER_RUN_PATH)
)

(
    quarantine_final_df.write
    .format("delta")
    .mode("overwrite")
    .save(QUARANTINE_RUN_PATH)
)

print("Silver saved to:", SILVER_RUN_PATH)
print("Quarantine saved to:", QUARANTINE_RUN_PATH)


In [0]:
#Read Back and Verify the Saved Data

silver_readback_df = (
    spark.read
    .format("delta")
    .load(SILVER_RUN_PATH)
)

quarantine_readback_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_RUN_PATH)
)

silver_saved_count = silver_readback_df.count()
quarantine_saved_count = quarantine_readback_df.count()

print("Silver rows saved:", f"{silver_saved_count:,}")
print("Quarantine rows saved:", f"{quarantine_saved_count:,}")

display(silver_readback_df.limit(5))


In [0]:
from pyspark.sql import functions as F

# Prepare the current run for the unified Silver table
readings_to_save = silver_final_df.withColumn(
    "run_id",
    F.lit(RUN_ID)
)

print("Current run:", RUN_ID)
print("Prepared readings:", readings_to_save.count())

In [0]:

from delta.tables import DeltaTable

SCHEMA = "workspace.default"

# 1. Create or incrementally update the unified readings table
READINGS_TABLE = f"{SCHEMA}.silver_smart_meter_readings"

if not spark.catalog.tableExists(READINGS_TABLE):
    (
        readings_to_save.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(READINGS_TABLE)
    )
else:
    target = DeltaTable.forName(spark, READINGS_TABLE)

    (
        target.alias("t")
        .merge(
            readings_to_save.dropDuplicates(["reading_id"]).alias("s"),
            "t.reading_id = s.reading_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 2. Create dimension tables without overwriting existing tables
dimension_tables = {
    "silver_dim_meter": dim_meter_df,
    "silver_dim_transformer": dim_transformer_df,
    "silver_dim_substation": dim_substation_df,
    "silver_dim_region": dim_region_df,
}

for table_name, df in dimension_tables.items():
    full_name = f"{SCHEMA}.{table_name}"

    if not spark.catalog.tableExists(full_name):
        (
            df.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(full_name)
        )
        print("Created:", full_name)
    else:
        print("Already exists; left unchanged:", full_name)

print("Readings table ready:", READINGS_TABLE)


In [0]:
READINGS_TABLE = "workspace.default.silver_smart_meter_readings"

spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT reading_id) AS distinct_readings,
    COUNT(DISTINCT run_id) AS total_runs,
    MIN(event_timestamp) AS first_reading,
    MAX(event_timestamp) AS last_reading
FROM {READINGS_TABLE}
""").show(truncate=False)

In [0]:
for table in [
    "silver_dim_meter",
    "silver_dim_transformer",
    "silver_dim_substation",
    "silver_dim_region",
]:
    full_name = f"workspace.default.{table}"

    print(f"\n{table}")
    spark.sql(
        f"SELECT COUNT(*) AS total_rows FROM {full_name}"
    ).show()